# Exploratory Data Analysis: `schools_data.csv`

This notebook performs EDA on Singapore primary school features, validates data quality, and uses plots to support key findings.

Key questions:
- What is the structure and quality of the dataset?
- Which variables have the strongest relationship with `top_psle_score`?
- Which schools have the highest admission demand pressure?


In [ ]:
# Install dependencies if needed (safe to re-run).
# Important: this keeps the notebook self-contained on fresh environments.
%pip install -q pandas numpy matplotlib seaborn

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 6)
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

In [ ]:
# Resolve dataset path robustly so the notebook works from either project root
# or the school_scoring folder.
candidate_paths = [
    Path('schools_data.csv'),
    Path('school_scoring/schools_data.csv'),
]

for p in candidate_paths:
    if p.exists():
        data_path = p
        break
else:
    raise FileNotFoundError('Could not find schools_data.csv in expected locations.')

df = pd.read_csv(data_path)
df.head()

In [ ]:
# Basic structure checks
print(f'Shape: {df.shape[0]} rows x {df.shape[1]} columns')
print('\nColumns:')
print(df.columns.tolist())

print('\nData types:')
display(df.dtypes.to_frame('dtype'))

In [ ]:
# Data quality checks
missing = df.isna().sum().sort_values(ascending=False)
duplicate_school_names = df['school_name'].duplicated().sum()

print('Missing values per column:')
display(missing.to_frame('missing_count'))
print(f'Duplicate school_name rows: {duplicate_school_names}')

In [ ]:
# Important: postal_code is numeric-looking but should be treated as an identifier.
id_like_columns = ['postal_code']

numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in id_like_columns]
categorical_cols = [c for c in df.columns if c not in numeric_cols and c not in id_like_columns]

print('Numeric columns for analysis:')
print(numeric_cols)
print('\nCategorical columns:')
print(categorical_cols)

In [ ]:
# Numeric summary statistics
display(df[numeric_cols].describe().T)

In [ ]:
# Distribution of school types
nature_counts = df['nature_code'].value_counts()
display(nature_counts.to_frame('count'))

ax = sns.countplot(data=df, x='nature_code', order=nature_counts.index, palette='Set2')
ax.set_title('School Type Distribution')
ax.set_xlabel('Nature Code')
ax.set_ylabel('Count')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
# Binary indicator prevalence
binary_cols = ['sap_ind', 'autonomous_ind', 'gifted_ind', 'top_psle_score']
indicator_rates = df[binary_cols].mean().sort_values(ascending=False)
display((indicator_rates * 100).round(2).to_frame('percentage'))

ax = indicator_rates.mul(100).plot(kind='bar', color='teal')
ax.set_title('Binary Indicator Rates (%)')
ax.set_xlabel('Indicator')
ax.set_ylabel('Percentage')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
# Engineer total demand feature used in findings.
demand_cols = ['P1_demand', 'P2A_demand', 'P2B_demand', 'P2C_demand', 'P2CS_demand']
df['total_demand'] = df[demand_cols].sum(axis=1)

top10_demand = df.nlargest(10, 'total_demand')[['school_name', 'total_demand']].reset_index(drop=True)
display(top10_demand)

ax = sns.barplot(data=top10_demand, y='school_name', x='total_demand', palette='viridis')
ax.set_title('Top 10 Schools by Total Demand')
ax.set_xlabel('Total Demand')
ax.set_ylabel('School Name')
plt.tight_layout()
plt.show()

In [ ]:
# Demand distributions by phase
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for i, col in enumerate(demand_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color='steelblue')
    axes[i].set_title(f'{col} Distribution')

# Last panel for total_demand
sns.histplot(df['total_demand'], kde=True, ax=axes[-1], color='darkorange')
axes[-1].set_title('total_demand Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Compare demand by top_psle_score class.
# Important: class is imbalanced, so always check group sizes before interpreting.
group_sizes = df['top_psle_score'].value_counts().sort_index()
display(group_sizes.to_frame('count'))

ax = sns.boxplot(data=df, x='top_psle_score', y='total_demand', palette='Set3')
ax.set_title('Total Demand by top_psle_score')
ax.set_xlabel('top_psle_score')
ax.set_ylabel('total_demand')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix for core numeric variables
corr_cols = [
    'P1_demand', 'P2A_demand', 'P2B_demand', 'P2C_demand', 'P2CS_demand',
    'subject_count', 'distprog_count', 'cca_clubs', 'cca_others', 'cca_sports',
    'cca_uniformed', 'cca_arts', 'affiliation_count', 'sap_ind', 'autonomous_ind',
    'gifted_ind', 'total_demand', 'top_psle_score'
]
corr = df[corr_cols].corr(numeric_only=True)

plt.figure(figsize=(14, 11))
sns.heatmap(corr, cmap='coolwarm', center=0, linewidths=0.3)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

corr_with_target = corr['top_psle_score'].drop('top_psle_score').sort_values(ascending=False)
display(corr_with_target.to_frame('corr_with_top_psle_score'))

ax = corr_with_target.plot(kind='bar', color='slateblue')
ax.set_title('Correlation with top_psle_score')
ax.set_xlabel('Feature')
ax.set_ylabel('Pearson correlation')
plt.xticks(rotation=70)
plt.tight_layout()
plt.show()

In [ ]:
# top_psle_score rate by school type
nature_psle = pd.crosstab(df['nature_code'], df['top_psle_score'], normalize='index')
display((nature_psle * 100).round(2))

ax = nature_psle.plot(kind='bar', stacked=True, colormap='Accent')
ax.set_title('top_psle_score Composition by School Type')
ax.set_xlabel('nature_code')
ax.set_ylabel('Proportion')
plt.xticks(rotation=20)
plt.legend(title='top_psle_score', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Interpretation notes
- `gifted_ind`, `P2B_demand`, `total_demand`, and selected CCA features show stronger positive associations with `top_psle_score`.
- `top_psle_score` is rare, so treat correlations as directional evidence, not causal proof.
- Use this EDA as a basis for feature engineering and class-imbalance-aware modeling next.
